In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when, isnull, length, trim, lower
from pyspark.sql.functions import upper, trim, col
from pyspark.sql.functions import to_timestamp
file_path = "/Volumes/workspace/default/retail_data/online_retail_II.csv"
df_spark = spark.read.csv(file_path, header=True, inferSchema=True)
display(df_spark)
print("--- Data profiling: MISSING VALUES ---")
null_counts = df_spark.select([count(when(isnull(c), c)).alias(c) for c in df_spark.columns])
display(null_counts)
print("--- Data profiling: HIDDEN SPACES IN DESCRIPTION ---")
space_mess = df_spark.filter(length(col("Description")) != length(trim(col("Description"))))
display(space_mess.select("Description"))
print("--- Data profiling: SPELLING & CASE ANOMALIES ---")
messy_spelling = df_spark.groupBy("Description").count().orderBy("Description")
display(messy_spelling)
print("--- Data profiling: MATHEMATICAL ANOMALIES ---")
display(df_spark.select("Quantity", "Price").describe())
#Cleaning 
print("Initiating Data Transformation Pipeline...")
df_clean = df_spark
df_clean = df_clean.dropna(subset=["Customer ID"])
df_clean = df_clean.filter((col("Quantity") > 0) & (col("Price") > 0))
df_clean = df_clean.withColumn("Description", trim(col("Description")))
df_clean = df_clean.withColumn("Description", upper(col("Description")))
df_clean.filter(isnull(col("Description"))).count()  
total = df_clean.count()
distinct = df_clean.distinct().count()
df_clean = df_clean.dropDuplicates()
df_clean = df_clean.withColumn(
    "InvoiceDate",
    to_timestamp(col("InvoiceDate"), "M/d/yyyy H:mm")
)
df_clean = df_clean.dropna(subset=["Description"])
print("FINAL CLEANED ROW COUNT:", df_clean.count())
df_clean.printSchema()


Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,01-12-2010 08:26,2.55,17850,United Kingdom
536365,71053,WHITE METAL LANTERN,6,01-12-2010 08:26,3.39,17850,United Kingdom
536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,01-12-2010 08:26,2.75,17850,United Kingdom
536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,01-12-2010 08:26,3.39,17850,United Kingdom
536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,01-12-2010 08:26,3.39,17850,United Kingdom
536365,22752,SET 7 BABUSHKA NESTING BOXES,2,01-12-2010 08:26,7.65,17850,United Kingdom
536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,01-12-2010 08:26,4.25,17850,United Kingdom
536366,22633,HAND WARMER UNION JACK,6,01-12-2010 08:28,1.85,17850,United Kingdom
536366,22632,HAND WARMER RED POLKA DOT,6,01-12-2010 08:28,1.85,17850,United Kingdom
536368,22960,JAM MAKING SET WITH JARS,6,01-12-2010 08:34,4.25,13047,United Kingdom


--- Data profiling: MISSING VALUES ---


Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,0,1454,0,0,0,135080,0


--- Data profiling: HIDDEN SPACES IN DESCRIPTION ---


Description
POPPY'S PLAYHOUSE BEDROOM
IVORY KNITTED MUG COSY
BOX OF VINTAGE JIGSAW BLOCKS
ALARM CLOCK BAKELIKE RED
STARS GIFT TAPE
INFLATABLE POLITICAL GLOBE
VINTAGE HEADS AND TAILS CARD GAME
SET/2 RED RETROSPOT TEA TOWELS
ROUND SNACK BOXES SET OF4 WOODLAND
SPACEBOY LUNCH BOX


--- Data profiling: SPELLING & CASE ANOMALIES ---


Description,count
null,1454
4 PURPLE FLOCK DINNER CANDLES,41
50'S CHRISTMAS GIFT BAG LARGE,130
DOLLY GIRL BEAKER,181
I LOVE LONDON MINI BACKPACK,88
I LOVE LONDON MINI RUCKSACK,1
NINE DRAWER OFFICE TIDY,34
OVAL WALL MIRROR DIAMANTE,162
RED SPOT GIFT BAG LARGE,105
SET 2 TEA TOWELS I LOVE LONDON,282


--- Data profiling: MATHEMATICAL ANOMALIES ---


summary,Quantity,Price
count,541910,541910
mean,9.552233765754462,4.611138332934584
stddev,218.08095694392347,96.75976549366496
min,-80995,-11062.06
max,80995,38970.0


Initiating Data Transformation Pipeline...
FINAL CLEANED ROW COUNT: 392693
root
 |-- Invoice: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- Price: double (nullable = true)
 |-- Customer ID: integer (nullable = true)
 |-- Country: string (nullable = true)

